# Phase 4.8 — Constructor `weight_by("optimize")`

Demonstrate the new `optimize` scheme on the fluent `Constructor` API: run the
Phase 4.1 convex optimizer (mean-variance) inside `Constructor.build()` and
compare against the `equal`-weight baseline on the same selection. The
full `OptimizationResult` is attached to the returned `ConstructionReport`.

In [1]:
from datetime import date
from unittest.mock import MagicMock

import numpy as np
import pandas as pd

from ah_research.backtest.types import Signals
from ah_research.portfolio.constructor import Constraint, Constructor
from ah_research.portfolio.optimizer import InfeasibleError, Optimizer
from ah_research.portfolio.optimizer.estimators.covariance import LedoitWolfCovariance
from ah_research.portfolio.optimizer.estimators.returns import UserSuppliedReturns

## Synthetic universe

We build an 8-symbol synthetic universe with 300 days of returns so the
covariance estimator has enough history. A `MagicMock` stands in for a full
`DataRepository` (the optimizer only calls `.get_prices(symbols, start, end)`).

In [2]:
symbols = [
    "600519.SH",
    "000858.SZ",
    "601398.SH",
    "600036.SH",
    "000333.SZ",
    "600276.SH",
    "600887.SH",
    "601318.SH",
]
rng = np.random.default_rng(42)
n_days = 300
dates = pd.bdate_range("2024-01-01", periods=n_days)
ret_matrix = rng.normal(0.0005, 0.015, size=(n_days, len(symbols)))
rows = []
for i, d in enumerate(dates):
    for j, s in enumerate(symbols):
        rows.append({"ds": d, "symbol": s, "close_hfq": 100.0, "total_return": ret_matrix[i, j]})
prices_df = pd.DataFrame(rows)

repo = MagicMock()


def _get_prices(syms, start, end):
    return prices_df[
        (prices_df["ds"] >= pd.Timestamp(start)) & (prices_df["ds"] <= pd.Timestamp(end))
    ].copy()


repo.get_prices.side_effect = _get_prices
len(prices_df)

2400

In [3]:
asof = date(2024, 12, 30)
signals_df = pd.DataFrame(
    {
        "date": pd.to_datetime([asof] * len(symbols)),
        "symbol": symbols,
        "signal": [1.0, 0.8, 0.6, 0.9, 0.4, 0.7, 0.3, 0.5],
    }
)
signals = Signals.from_dataframe(signals_df)
signals.df.head()

,date,symbol,signal
0,2024-12-30,600519.SH,1.0
1,2024-12-30,000858.SZ,0.8
2,2024-12-30,601398.SH,0.6
3,2024-12-30,600036.SH,0.9
4,2024-12-30,000333.SZ,0.4


## Baseline: `weight_by("equal")`

In [4]:
report_equal = (
    Constructor(signals, repo=repo, asof=asof).method("all_positive").weight_by("equal").build()
)
report_equal.weights

,symbol,weight
0,600519.SH,0.125
1,000858.SZ,0.125
2,601398.SH,0.125
3,600036.SH,0.125
4,000333.SZ,0.125
5,600276.SH,0.125
6,600887.SH,0.125
7,601318.SH,0.125


## New: `weight_by("optimize")` — mean-variance

Pass an `Optimizer` to `Constructor(optimizer=...)` and choose `weight_by("optimize")`.
We supply a user mu (so the demo is deterministic and fast) via `UserSuppliedReturns`.

In [5]:
mu = pd.Series(
    [0.012, 0.010, 0.008, 0.011, 0.005, 0.009, 0.004, 0.007],
    index=symbols,
)
optimizer = Optimizer(
    objective="mean_variance",
    cov_estimator=LedoitWolfCovariance(),
    returns_estimator=UserSuppliedReturns(mu),
    risk_aversion=5.0,
)
report_opt = (
    Constructor(signals, repo=repo, asof=asof, optimizer=optimizer)
    .method("all_positive")
    .weight_by("optimize")
    .build()
)
report_opt.weights

,symbol,weight
0,600519.SH,7.191658e-01
1,000858.SZ,6.782577e-23
2,601398.SH,-5.002052e-24
3,600036.SH,2.808342e-01
4,000333.SZ,5.055489e-23
5,600276.SH,7.824938e-23
6,600887.SH,-1.159633e-22
7,601318.SH,-4.986789e-24


## Side-by-side comparison

In [6]:
compare = pd.DataFrame(
    {
        "equal": report_equal.weights.set_index("symbol")["weight"],
        "optimize": report_opt.weights.set_index("symbol")["weight"],
    }
).round(4)
compare

,equal,optimize
symbol,,
600519.SH,0.125,0.7192
000858.SZ,0.125,0.0000
601398.SH,0.125,-0.0000
600036.SH,0.125,0.2808
000333.SZ,0.125,0.0000
600276.SH,0.125,0.0000
600887.SH,0.125,-0.0000
601318.SH,0.125,-0.0000


## `OptimizationResult` markdown summary

The `ConstructionReport.optimization_result` field carries the full
`OptimizationResult` (solver status, active constraints, risk contributions,
slack, inputs hash, timing).

In [7]:
assert report_opt.optimization_result is not None
print(report_opt.optimization_result.to_markdown())

# Optimization Result (mean_variance)

- **Solver:** osqp (optimal)
- **Objective value:** -0.0110392
- **Expected variance:** 0.000135985
- **Expected return:** 0.0117192
- **Active constraints:** long_only
- **Solve time:** 4.4 ms
- **Inputs hash:** `427aa533ccd6…`

## Weights

| Symbol | Weight |
|---|---|
| 600519.SH | 0.7192 |
| 600036.SH | 0.2808 |
| 600276.SH | 0.0000 |
| 000858.SZ | 0.0000 |
| 000333.SZ | 0.0000 |
| 601318.SH | -0.0000 |
| 601398.SH | -0.0000 |
| 600887.SH | -0.0000 |


## Infeasible path: very tight `max_weight`

When constraints are physically impossible (here: cap each weight at 5% across
8 symbols — sum cannot reach 1), the optimizer raises `InfeasibleError`.

In [8]:
tight_optimizer = Optimizer(
    objective="mean_variance",
    cov_estimator=LedoitWolfCovariance(),
    returns_estimator=UserSuppliedReturns(mu),
    constraints=[Constraint.max_weight(0.05)],
    risk_aversion=5.0,
)
try:
    (
        Constructor(signals, repo=repo, asof=asof, optimizer=tight_optimizer)
        .method("all_positive")
        .weight_by("optimize")
        .build()
    )
    print("Unexpectedly feasible!")
except InfeasibleError as e:
    print(f"InfeasibleError (as expected): {e}")

InfeasibleError (as expected): Problem is infeasible. Use soft=True for best-effort.


## Self-checks

In [9]:
assert report_equal.weighting_scheme == "equal"
assert report_opt.weighting_scheme == "optimize"
assert abs(report_equal.weights["weight"].sum() - 1.0) < 1e-6
assert abs(report_opt.weights["weight"].sum() - 1.0) < 1e-4
assert report_opt.optimization_result is not None
assert report_opt.optimization_result.solver_status in ("optimal", "optimal_inaccurate")
print("All checks passed")

All checks passed
